# Analyzing Experiment Results

Load experiments, inspect JSONL iteration data, and compare summary metrics across models.

In [ ]:
from collections import defaultdict
from statistics import mean

from faas_gauge.config import get_data_dir
from faas_gauge.notebooks.helpers import format_iteration_table, print_experiment_summary
from faas_gauge.store import ExperimentStore

data_dir = get_data_dir()
store = ExperimentStore(data_dir=data_dir)

In [ ]:
experiments = store.list_experiments(pattern="*", test_group="notebook-demo")
print("experiments found:", len(experiments))

for meta in experiments[:3]:
    print_experiment_summary(meta["id"], meta.get("summary", {}))

In [ ]:
if experiments:
    first_id = experiments[0]["id"]
    iteration_rows = list(store.read_iterations(first_id))
    print("sample JSONL rows:", len(iteration_rows))
    print(format_iteration_table(iteration_rows, max_rows=15))

In [ ]:
by_model = defaultdict(list)
for meta in experiments:
    summary = meta.get("summary", {})
    model = meta.get("model", "unknown")
    by_model[model].append(
        {
            "success_rate": (
                summary.get("successful_iterations", 0)
                / max(summary.get("total_iterations", 1), 1)
            ),
            "avg_time": summary.get("average_time", 0.0),
            "total_tokens": summary.get("total_input_tokens", 0)
            + summary.get("total_output_tokens", 0),
        }
    )

for model, rows in sorted(by_model.items()):
    print(f"Model: {model}")
    print("  experiments :", len(rows))
    print("  mean success:", round(mean(r["success_rate"] for r in rows), 3))
    print("  mean avg sec:", round(mean(r["avg_time"] for r in rows), 3))
    print("  total tokens:", sum(r["total_tokens"] for r in rows))

This notebook works directly with JSON metadata and JSONL iteration records from the data store, so you can build custom analysis pipelines in plain Python.